### Product Profitability

In [0]:
%sql
CREATE OR REPLACE TABLE gold.product_profitability
USING DELTA
AS
SELECT
  p.product_key,
  p.product_name,
  p.category,
  p.subcategory,
  p.product_line,
  p.cost,
 
  -- Volume metrics
  COUNT(DISTINCT s.order_number)                    AS total_orders,
  SUM(s.quantity)                                   AS units_sold,
 
  -- Revenue metrics
  SUM(s.sales_amount)                               AS total_revenue,
  AVG(s.sales_amount)                               AS avg_order_revenue,
 
  -- Profit = what we sold it for minus what it cost to make
  SUM(s.sales_amount - (p.cost * s.quantity))       AS total_profit,
  AVG(s.sales_amount - (p.cost * s.quantity))       AS avg_profit_per_order,
 
  -- Profit margin % — the most important number for management
  ROUND(
    SUM(s.sales_amount - (p.cost * s.quantity))
    / NULLIF(SUM(s.sales_amount), 0) * 100, 2
  )                                                 AS profit_margin_pct,
 
  -- Revenue share — what % of total company revenue does this product represent
  ROUND(
    SUM(s.sales_amount) * 100.0
    / SUM(SUM(s.sales_amount)) OVER (), 2
  )                                                 AS revenue_share_pct,
 
  -- Performance label — management can filter by this instantly
  CASE
    WHEN ROUND(SUM(s.sales_amount - (p.cost * s.quantity))
         / NULLIF(SUM(s.sales_amount), 0) * 100, 2) >= 40 THEN 'High Performer'
    WHEN ROUND(SUM(s.sales_amount - (p.cost * s.quantity))
         / NULLIF(SUM(s.sales_amount), 0) * 100, 2) >= 20 THEN 'Moderate Performer'
    WHEN ROUND(SUM(s.sales_amount - (p.cost * s.quantity))
         / NULLIF(SUM(s.sales_amount), 0) * 100, 2) >= 0  THEN 'Low Performer'
    ELSE 'Loss Making'
  END                                               AS performance_label
 
FROM workspace.silver.sales s
JOIN workspace.silver.products p
  ON s.product_key = p.product_key
 
GROUP BY
  p.product_key, p.product_name, p.category,
  p.subcategory, p.product_line, p.cost
 
ORDER BY total_profit DESC;


In [0]:
%sql
SELECT *
FROM workspace.gold.product_profitability
LIMIT 10;

Databricks visualization. Run in Databricks to view.